# Fair-v3 NVIDIA-layout 2:4: notebook chạy các thí nghiệm

Notebook này chạy độc lập trên Kaggle theo thứ tự cố định:

1. kiểm tra mã nguồn và dataset;
2. ISIC main-table baselines và mô hình tham chiếu;
3. component ablations bắt buộc;
4. softmax long-tail appendix;
5. CIFAR-100-LT 1:100 từ dataset Kaggle đã attach (không tải Internet);
6. OOD/PAD tùy chọn;
7. tổng hợp chỉ các run đúng protocol.

Các cell dừng ngay khi lỗi, tự bỏ qua run đã hoàn tất, và không dùng smoke test để tránh trộn kết quả 1 epoch với kết quả báo cáo.

## 0. Clone/pull và khóa phiên bản mã nguồn

Nếu notebook đã clone repo, cell dùng `git pull --ff-only`; nếu chưa có, cell clone mới. Hãy commit/push toàn bộ fair-v3 NVIDIA-layout source trước khi chạy notebook này.

In [ ]:
import os, sys, json, math, shutil, subprocess, shlex, time
from pathlib import Path

REPO_URL = "https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git"
REPO_ROOT = Path("/kaggle/working/MDEP-Microglial-Driven-Evidential-Pruning")

def run(cmd, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("\n$", shlex.join(cmd), flush=True)
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    subprocess.run(cmd, cwd=str(cwd or REPO_ROOT), env=merged, check=True)

if not (REPO_ROOT / ".git").exists():
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, REPO_ROOT], cwd="/kaggle/working")
else:
    run(["git", "pull", "--ff-only"], cwd=REPO_ROOT)

os.chdir(REPO_ROOT)
os.environ.update({
    "MDEP_DETERMINISTIC": "1",
    "PYTHONHASHSEED": "0",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
})

required = [
    "guds_edl_core.py",
    "experiments/isic_paper_experiments.py",
    "experiments/generalization_paper_suite.py",
    "experiments/cifar_lt_runner.py",
    "experiments/run_external_validation.py",
]
missing = [p for p in required if not (REPO_ROOT / p).exists()]
assert not missing, f"Thiếu source files: {missing}"
runner_text = (REPO_ROOT / "experiments/isic_paper_experiments.py").read_text(encoding="utf-8")
PROTOCOL = "isic_fair_v3_nvidia24_2026_07_09"
assert PROTOCOL in runner_text, "Repo hiện tại chưa chứa fair-v3 NVIDIA-layout protocol. Hãy push/pull source mới trước khi train."
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
print(f"[OK] repo={REPO_ROOT}\n[OK] commit={commit}\n[OK] protocol={PROTOCOL}")
run([sys.executable, "-c", "import torch, torchvision, pandas, sklearn, h5py; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"])


## 1. Tìm dataset trên Kaggle

Trước khi chạy, attach:

- ISIC 2024 competition data, có `train-metadata.csv` và `train-image.hdf5`;
- một Kaggle dataset chứa bản gốc `cifar-100-python/train`, `test`, `meta`;
- PAD-UFES-20 chỉ khi bật OOD/PAD.

CIFAR được đọc thẳng từ `/kaggle/input` bằng biến `CIFAR_ROOT`; cell không gọi download.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
assert INPUT_ROOT.exists(), "Cell này được thiết kế cho Kaggle Notebook."

def first_valid(candidates, predicate):
    for p in candidates:
        try:
            if predicate(p):
                return p.resolve()
        except OSError:
            pass
    return None

isic_candidates = [
    INPUT_ROOT / "isic-2024-challenge",
    INPUT_ROOT / "competitions/isic-2024-challenge",
] + [p.parent for p in INPUT_ROOT.rglob("train-metadata.csv")]
ISIC_ROOT = first_valid(
    isic_candidates,
    lambda p: (p / "train-metadata.csv").exists() and ((p / "train-image.hdf5").exists() or (p / "train-image/image").exists()),
)
assert ISIC_ROOT, "Không tìm thấy ISIC 2024. Hãy Add Data/competition ISIC 2024."
os.environ["ISIC_ROOT"] = str(ISIC_ROOT)

cifar_folders = list(INPUT_ROOT.rglob("cifar-100-python"))
CIFAR_FOLDER = first_valid(
    cifar_folders,
    lambda p: all((p / name).exists() for name in ("train", "test", "meta")),
)
assert CIFAR_FOLDER, (
    "Không tìm thấy CIFAR bản Python gốc. Attach Kaggle dataset có "
    "cifar-100-python/{train,test,meta}; không dùng dataset CSV/PNG cho protocol này."
)
os.environ["CIFAR_ROOT"] = str(CIFAR_FOLDER.parent)

pad_csvs = [p for p in INPUT_ROOT.rglob("metadata.csv") if "skin" in str(p).lower() or "pad" in str(p).lower()]
PAD_META = pad_csvs[0].resolve() if pad_csvs else None
PAD_ROOT = PAD_META.parent if PAD_META else None

print("[OK] ISIC_ROOT =", ISIC_ROOT)
print("[OK] CIFAR_ROOT =", os.environ["CIFAR_ROOT"], "(download=False sẽ được dùng)")
print("[INFO] PAD_ROOT =", PAD_ROOT or "không attach; chỉ cần nếu RUN_OOD=True")


## 2. Cấu hình chạy và cơ chế resume

`RUN_*` có thể đổi trước khi chạy. Mặc định chạy toàn bộ phần còn thiếu cốt lõi. `RUN_OOD` chỉ bật nếu PAD đã attach. Layer-4 KD không thuộc so sánh chính và được để tắt.

In [ ]:
SEEDS = [42, 123, 456]
SPLIT_SEED = 42
ISIC_EPOCHS = 40
RUN_MAIN_TABLE = True
RUN_REQUIRED_ABLATIONS = True
RUN_OPTIONAL_DIAGNOSTIC_ABLATIONS = False
RUN_SOFTMAX_APPENDIX = True
RUN_CIFAR100_LT = True
RUN_OOD = PAD_ROOT is not None
RUN_LAYER4_KD = False

OUT = Path("/kaggle/working/paper_experiment_outputs")
ISIC_OUT = OUT / "isic"
FAIR_SUFFIX = "_fair_v3_nvidia24"
CIFAR_SUFFIX = "_fair_v3_nvidia24_2026_07_09"

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def isic_seed_valid(name, seed, require_model=False):
    d = ISIC_OUT / f"{name}{FAIR_SUFFIX}" / f"seed_{seed}"
    p = d / "metrics.json"
    if not p.exists() or (require_model and not (d / "model_state.pth").exists()):
        return False
    try:
        x = read_json(p)
        return x.get("protocol_version") == PROTOCOL and x.get("epochs") == ISIC_EPOCHS and x.get("split_seed") == SPLIT_SEED
    except Exception:
        return False

def missing_isic(names, require_model=False):
    return [n for n in names if not all(isic_seed_valid(n, s, require_model=require_model and n == "full_guds") for s in SEEDS)]

def run_isic(names, save_model=False):
    ran_any = False
    for seed in SEEDS:
        pending = [n for n in names if not isic_seed_valid(n, seed, require_model=save_model)]
        if not pending:
            print(f"[SKIP] seed={seed}: đã có đủ {names}")
            continue
        ran_any = True
        cmd = [sys.executable, "-u", "experiments/isic_paper_experiments.py"]
        for name in pending:
            cmd += ["--experiment", name]
        cmd += [
            "--epochs", str(ISIC_EPOCHS), "--batch_size", "32", "--lr", "4e-5",
            "--seed", str(seed), "--split_seed", str(SPLIT_SEED),
            "--subsample_scope", "train", "--subsample_ratio", "20",
            "--structural_proxy_batches", "4", "--checkpoint_selection", "last",
            "--run_suffix", FAIR_SUFFIX,
        ]
        if not save_model:
            cmd.append("--no_save_model")
        run(cmd)
    if not ran_any:
        print("[SKIP] Nhóm này đã hoàn tất đủ ba seed.")

def gate_isic(names, require_model=False):
    bad = [(n, s) for n in names for s in SEEDS if not isic_seed_valid(n, s, require_model=require_model)]
    assert not bad, f"Quality gate thất bại, thiếu/sai protocol: {bad}"
    print(f"[GATE PASS] {len(names)} experiments × {len(SEEDS)} seeds")

print({k: v for k, v in globals().copy().items() if k.startswith("RUN_")})


## 3. Main table: baseline trước, DST-EDL tham chiếu sau

Thứ tự này hoàn tất sáu baseline evidential/sparse trước. `full_guds` được lưu checkpoint vì OOD cần checkpoint; baseline chỉ lưu metrics/predictions để tiết kiệm dung lượng.

In [ ]:
MAIN_BASELINES = ["fisher_edl", "flexible_edl", "r_edl"]
CHECKPOINTED_BASELINES = ["dense_edl", "static_24_edl", "rigl_style_24"]
MAIN_REFERENCE = ["full_guds"]

if RUN_MAIN_TABLE:
    run_isic(MAIN_BASELINES, save_model=False)
    gate_isic(MAIN_BASELINES)
    run_isic(CHECKPOINTED_BASELINES, save_model=True)
    gate_isic(CHECKPOINTED_BASELINES, require_model=True)
    run_isic(MAIN_REFERENCE, save_model=True)
    gate_isic(MAIN_REFERENCE, require_model=True)
else:
    print("[SKIP] RUN_MAIN_TABLE=False")


## 4. Component ablations bắt buộc

Năm ablation dưới đây trả lời trực tiếp contribution của pruner, regrower, symmetric KL, EFL và anti-crystallization. Các diagnostic mở rộng được tách riêng để không tiêu GPU trước khi bảng ablation chính hoàn tất.

In [ ]:
REQUIRED_ABLATIONS = [
    "guds_without_pruner", "guds_without_regrower", "guds_asymmetric_kl",
    "guds_without_efl", "guds_without_anticryst",
]
OPTIONAL_ABLATIONS = [
    "guds_absolute_pruner", "guds_class_conditioned_regrower",
    "guds_without_topology_cache", "guds_temperature_only", "guds_no_posthoc_calibration",
]

if RUN_REQUIRED_ABLATIONS:
    gate_isic(MAIN_REFERENCE, require_model=True)
    run_isic(REQUIRED_ABLATIONS, save_model=False)
    gate_isic(REQUIRED_ABLATIONS)
if RUN_OPTIONAL_DIAGNOSTIC_ABLATIONS:
    run_isic(OPTIONAL_ABLATIONS, save_model=False)
    gate_isic(OPTIONAL_ABLATIONS)


## 5. Softmax long-tail appendix

Các mô hình này dùng probability/softmax evaluator, nhưng giữ nguyên split, seed, epoch, batch size và preprocessing để so sánh công bằng.

In [ ]:
SOFTMAX_BASELINES = [
    "standard_ce", "focal_loss", "class_balanced_ce", "balanced_softmax",
    "ldam_drw", "decoupled_crt", "mislas",
]
if RUN_SOFTMAX_APPENDIX:
    run_isic(SOFTMAX_BASELINES, save_model=False)
    gate_isic(SOFTMAX_BASELINES)


## 6. CIFAR-100-LT 1:100 từ Kaggle

Đây là diagnostic ngoài claim chính: Standard CE, Dense EDL và DST-EDL, 100 epoch, ba model seed, cùng một `split_seed=42`. Suffix riêng ngăn ghi đè kết quả cũ.

In [ ]:
CIFAR_MODELS = ["standard_ce", "dense_edl", "full_guds"]
CIFAR_OUT = OUT / "cifar" / "ir100"

def cifar_seed_valid(name, seed):
    p = CIFAR_OUT / f"{name}{CIFAR_SUFFIX}" / f"seed_{seed}" / "metrics.json"
    if not p.exists():
        return False
    try:
        x = read_json(p)
        e = x.get("experiment", {})
        return x.get("epochs") == 100 and x.get("split_seed") == SPLIT_SEED and e.get("run_suffix") == CIFAR_SUFFIX
    except Exception:
        return False

if RUN_CIFAR100_LT:
    for seed in SEEDS:
        missing_cifar = [n for n in CIFAR_MODELS if not cifar_seed_valid(n, seed)]
        if not missing_cifar:
            print(f"[SKIP] CIFAR seed={seed}: đủ ba mô hình")
            continue
        cmd = [sys.executable, "-u", "experiments/generalization_paper_suite.py", "--benchmark", "cifar"]
        for name in missing_cifar:
            cmd += ["--experiment", name]
        cmd += [
            "--ratio", "100", "--epochs", "100", "--batch_size", "128",
            "--seed", str(seed), "--split_seed", str(SPLIT_SEED),
            "--run_suffix", CIFAR_SUFFIX, "--log_every", "5",
        ]
        run(cmd, env={"CIFAR_ROOT": os.environ["CIFAR_ROOT"]})
    bad = [(n, s) for n in CIFAR_MODELS for s in SEEDS if not cifar_seed_valid(n, s)]
    assert not bad, f"CIFAR quality gate thất bại: {bad}"
    print("[GATE PASS] CIFAR-100-LT 1:100: 3 models × 3 seeds; split_seed=42; epochs=100")


## 7. OOD trên PAD-UFES-20 (sau khi main/ablation hoàn tất)

OOD dùng checkpoint fair-v3 NVIDIA-layout của từng seed và `knn_layer3` làm primary score. `imgs_part_3` chỉ là OOD holdout, không dùng để tuyên bố PAD classification/fairness.

In [ ]:
if RUN_OOD:
    assert PAD_ROOT and PAD_META and PAD_META.exists(), "RUN_OOD=True nhưng chưa attach PAD-UFES-20."
    gate_isic(MAIN_REFERENCE, require_model=True)
    for seed in SEEDS:
        checkpoint = ISIC_OUT / "full_guds_fair_v3_nvidia24" / f"seed_{seed}" / "model_state.pth"
        run([
            sys.executable, "-u", "experiments/run_external_validation.py",
            "--model_path", checkpoint, "--seed", seed, "--split_seed", SPLIT_SEED,
            "--custom_image_folder", PAD_ROOT, "--pad_ufes_csv", PAD_META,
            "--pad_ufes_partition", "imgs_part_3", "--knn_primary_layer", "layer3",
            "--primary_ood_score", "knn_layer3",
        ])
else:
    print("[SKIP] OOD: PAD chưa attach hoặc RUN_OOD=False")

assert not RUN_LAYER4_KD, (
    "Layer-4 KD được cố ý loại khỏi notebook so sánh chính. Chỉ bật/chạy trong một analysis riêng "
    "sau khi frozen-adapter protocol thất bại và phải ghi rõ đây là adaptation, không phải baseline công bằng."
)


## 8. Tổng hợp ba seed và xuất manifest

Cell chỉ đọc thư mục có suffix/protocol đã khóa, tính mean và sample std (`ddof=1`), đồng thời lưu commit và dataset paths.

In [ ]:
import pandas as pd
import numpy as np

all_isic = MAIN_BASELINES + CHECKPOINTED_BASELINES + MAIN_REFERENCE + REQUIRED_ABLATIONS + SOFTMAX_BASELINES
if RUN_OPTIONAL_DIAGNOSTIC_ABLATIONS:
    all_isic += OPTIONAL_ABLATIONS

rows = []
for name in all_isic:
    for seed in SEEDS:
        p = ISIC_OUT / f"{name}{FAIR_SUFFIX}" / f"seed_{seed}" / "metrics.json"
        if not p.exists():
            continue
        x = read_json(p)
        if x.get("protocol_version") != PROTOCOL or x.get("epochs") != ISIC_EPOCHS:
            continue
        m = x.get("metrics", {})
        rows.append({
            "dataset": "ISIC", "experiment": name, "seed": seed,
            "pauc": m.get("pauc"), "pr_auc": m.get("pr_auc"),
            "balanced_accuracy": m.get("balanced_accuracy"),
            "ece_adaptive": m.get("ece_adaptive"), "aurc": m.get("aurc"),
        })

for name in CIFAR_MODELS:
    for seed in SEEDS:
        p = CIFAR_OUT / f"{name}{CIFAR_SUFFIX}" / f"seed_{seed}" / "metrics.json"
        if not p.exists():
            continue
        x = read_json(p)
        if x.get("epochs") != 100 or x.get("split_seed") != SPLIT_SEED:
            continue
        m = x.get("metrics", {})
        rows.append({
            "dataset": "CIFAR100LT_IR100", "experiment": name, "seed": seed,
            "balanced_accuracy": m.get("balanced_accuracy"),
            "macro_auroc": m.get("macro_auroc"), "macro_pr_auc": m.get("macro_pr_auc"),
            "ece_adaptive": m.get("ece_adaptive"), "aurc": m.get("aurc"),
        })

seed_df = pd.DataFrame(rows)
assert not seed_df.empty, "Chưa có metrics hợp lệ để tổng hợp."
metric_cols = [c for c in seed_df.columns if c not in {"dataset", "experiment", "seed"}]
summary = seed_df.groupby(["dataset", "experiment"])[metric_cols].agg(["mean", "std", "count"])
SUMMARY_DIR = OUT / "fair_v3_nvidia24_reports"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
seed_df.to_csv(SUMMARY_DIR / "seed_level_metrics.csv", index=False)
summary.to_csv(SUMMARY_DIR / "three_seed_summary.csv")

manifest = {
    "protocol": PROTOCOL, "git_commit": commit, "seeds": SEEDS, "split_seed": SPLIT_SEED,
    "isic_root": str(ISIC_ROOT), "cifar_root": os.environ["CIFAR_ROOT"],
    "pad_root": str(PAD_ROOT) if PAD_ROOT else None,
    "notes": [
        "ISIC summary accepts only fair-v3 NVIDIA-layout protocol and 40 epochs.",
        "CIFAR summary accepts only IR100, 100 epochs, split seed 42, and dedicated suffix.",
        "PAD imgs_part_3 is OOD-only; no PAD classification/fairness claim.",
    ],
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
display(summary)
print("\n[DONE] Reports:", SUMMARY_DIR)
print("-", SUMMARY_DIR / "seed_level_metrics.csv")
print("-", SUMMARY_DIR / "three_seed_summary.csv")
print("-", SUMMARY_DIR / "run_manifest.json")
